# **Kurs: 4050 - Photogrammetrie, Computer Vision & English**

**Autoren:**  
Jonas Meyer  
 Elia Ferrari  
 Prof. Dr. Stephan Nebiker

Datum: 15.04.2024

# **Uebung Einzelbildorientirung mit DLT**

### Einleitung und Zielsetzung

In dieser Übung lernst du eine Möglichkeit kennen, um mithilfe der Beziehung von Bild- und Passpunkten Näherungswerte der inneren und äusseren Orientierungsparameter eines Bildes zu bestimmen. Die Bestimmung der Näherungswerte soll mit der direkten linearen Transformation (DLT) aus dem Bereich der Computer Vision am Beispiel der Einzelbildorientierung durchgeführt werden.

Am Schluss dieser Übung solltest du: 

* Die Näherungswerte der inneren und äusseren Orientierungsparameter mit einer direkten linearen Transformation (DLT) bestimmen können
* Basierend auf den Näherungwerten der DLT den Rückwärtsschnitt eines Einzelbildes rechnen können.
* Die Resultate auf Plausibilität überprüfen können.


### Benötigte Module

* Scikit Image
* Numpy
* Matplotlib

### Abgabe

Jede Studentin und jeder Student muss ein funktionierendes Code-Repository rechtzeitig auf GitHub (via push-Funktion) abgeben.
Die Funktionstüchtigkeit des eigenen Codes ist sowohl mit Hilfe der vorhandenen unittests sowie anhand der Resultate vorgängig zu beurteilen.

### Vorgehen

Die Übung umfasst die folgenden Schritte:
1.	Definition der Bild- und Objektkoordinaten
2.	Implementierung der direkten linearen Transformation (DLT)
3.  Berechnung Näherungswerte mittels CLT
4.  Berechnung äussere Orientierung mittels räumlichem Rückwärtsschnitt
5.  Visualisierung der Resultate

### Zur Info

* Zu ergänzende Code-Stellen sind mit ``#TODO`` gekennnzeichnet.
* Bei Unklarheiten bezüglich der Bedeutung englischer Begriffe aus dem Bereich Photogrammetrie kann [dieses Dokument](https://onlinelibrary.wiley.com/doi/10.1111/phor.12314) konsultiert werden.
* Für die Durchführung der unittests siehe Anleitung in [README.md](README.md).

### Import notwendige Module und Daten

Erforderliche Module

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io

from src.utils import *
from src.rotation_matrix import *
from src.collinearity_equation import *
from src.direct_linear_transform import *

## **Schritt 1**

### Datenimport

Lade das Bild `DJI_0149.jpg` aus dem Verzeichnis `data/` und weise es der Variable `img` zu.

In [3]:
img = io.imread('data/DJI_0149.jpg')

### Definition der Bild- und Objektkoordinaten
#### Manuelle Messung Bildkoordinaten

Vervollständige den folgenden Code und messe die Bildpunkte der Reihenfolge nach (Punktnummer aufsteigend). 

<figure align="center">
<img src="data/Passpunkte.png" alt="Bildaufnahme mit eingezeichnete Passpunkten" width="1000"/>
<figcaption>Abb. 1 - Bildaufnahme mit eingezeichnete Passpunkten.</figcaption>
</figure>


Konvertiere die resultierende Liste zu einem ``numpy.ndarray`` und speichere die Sensorkoordinaten in die Variabeln ``u`` und ``v``.

In [4]:
%matplotlib qt

plt.imshow(img)

points = 12
plt.title(f'Klick all {points} points in the order of their number', fontweight ="bold")
pts = plt.ginput(n=points, timeout=0)
plt.close()

In [5]:
image_pts = np.array(pts)

u = image_pts[:, 0]
v = image_pts[:, 1]

#### Umwandlung Sensor- zu Bildkoordinaten

Die gemessenen Sensorkoordinaten ``u`` und ``v`` liegen in der Einheit Pixel vor. Konvertiere die Sensorkoordinaten in Bildkoordinaten mit der Einheit Millimeter (siehe Bild unten). Für die Umrechnung sind sowohl die Pixelgrösse für das Ausgangsbild (0.00241 mm) als auch die Sensorgrösse notwendig. Frage die Sensorgrösse (Anzahl Pixel Höhe / Breite des Bildes) mit einer geeigneten Funktion ab.

<figure align="center">
<img src="docs/sensor_image_coordinate_system.PNG" alt="Sensor- und Bildkoordinatensystem" width="400"/>
<figcaption>Abb. 2 - Definition Sensor- und Bildkoordinatensytem (Luhmann, 2018).</figcaption>
</figure>

In [6]:
pix_size_mm = 0.00241
size_v, size_u , c = img.shape
x_measured_mm = (u-size_u / 2)*pix_size_mm
y_measured_mm = -(v-size_v / 2)*pix_size_mm

#### Definition Objektkoordinaten

Lade die Liste der Objektkoordinaten mithilfe der Funktion [np.loadtxt](https://numpy.org/doc/stable/reference/generated/numpy.loadtxt.html), beachte dabei das Trennzeichen und den Header der Datei. Fülle anschliessend die Variabeln ``pt_nr``, ``X``, ``Y``, ``Z`` und ``num_pts`` ab.

In [7]:
obj_pts_m = np.loadtxt('data/passpunkte.txt', delimiter=';', skiprows=1)

pt_nr = obj_pts_m[: ,0]
X = obj_pts_m[: , 1]
Y = obj_pts_m[: , 2]
Z = obj_pts_m[: , 3]
num_pts = obj_pts_m.shape[0]

## **Schritt 2**

#### Direkte lineare Transformation (Näherungswertbestimmung) ####

Die direkte lineare Transformation nutzt, wie der Name bereits sagt, die lineare Beziehung zwischen den Bild- und den Objektkoordinaten und ist deshalb stark an die Kollinearitätsgleichung der Photogrammetrie angelehnt. Dieser Algorithmus stammt aus dem Bereich der Computer Vision. Die Bestimmung der Parameter der direkten linearen Transformation erfolgt durch eine Ausgleichung nach der Methode der kleinsten Quadrate. Für die Ausgleichung werden die ``A-Matrix`` und den ``f-Vektor`` benötigt. Ergänze in der Datei [src/direct_linear_transform.py](src/direct_linear_transform.py) zunächst die Initilisierung der A-Matrix und des f-Vektors. Anschliessend sollen die Koeffizienten der A-Matrix analog der unten aufgeführten Formel ergänzt werden.

$$ A =\begin{bmatrix} X_1 & Y_1 & Z_1 & 1 & 0 & 0 & 0 & 0 & -x_1 X_1 & -x_1 Y_1 & -x_1 Z_1\\
                    0 & 0 & 0 & 0 & X_1 & Y_1 & Z_1 & 1 & -y_1 X_1 & -y_1 Y_1 & -y_1 Z_1 \\
                    X_2 & Y_2 & Z_2 & 1 & 0 & 0 & 0 & 0 & -x_2 X_2 & -x_2 Y_2 & -x_2 Z_2\\
                    0 & 0 & 0 & 0 & X_2 & Y_2 & Z_2 & 1 & -y_2 X_2 & -y_2 Y_2 & -y_2 Z_2 \\
                    ... & ... & ... & ... & ... & ... & ... & ... & ... & ... & ...\\
                    X_n & Y_n & Z_n & 1 & 0 & 0 & 0 & 0 & -x_n X_n & -x_n Y_n & -x_n Z_n\\
                    0 & 0 & 0 & 0 & X_n & Y_n & Z_n & 1 & -y_n X_n & -y_n Y_n & -y_n Z_n \\\end{bmatrix};  f =\begin{bmatrix} x_1 \\ y_1 \\ x_2 \\ y_2 \\ ... \\ x_n \\ y_n \\\end{bmatrix} $$

Führe anschliessend die Ausgleichung nach der Methode der kleinsten Quadrate durch.

**Ausgleichung:**<br><br>

$x = (A^T A)^{-1} A^T f$

Ergänze die restlichen fehlenden Berechnungen für den DLT-Algorithmus gemäss den untenstehenden Formeln

**Definition Hilfgrösse L**<br><br>
$L= \frac{-1}{\sqrt{L_{9}^{2} + L_{10} ^2 + L_{11} ^2}}$

**Berechnung Parameter der inneren Orientierung**<br><br>
Bildhauptpunkt:<br>

$x_0 = L^2(L_1 L_9 + L_2 L_{10} + L_3 L_{11})$ <br>
$y_0 = L^2(L_5 L_9 + L_6 L_{10} + L_7 L_{11})$
<br><br>
Kamerakonstante (skaliert in x und y):<br>

$c_x = \sqrt{(L^2(L_1^2 + L_2^2 + L_3^2)-x_0^2)}$<br>
$c_y = \sqrt{(L^2(L_5^2 + L_6^2 + L_7^2)-y_0^2)}$<br>
$pd = \frac{(c_x + c_y)}{2}$
(pd steht für principal distance und ist dasselbe $c_k$)

**Berechnung Parameter der äusseren Orientierung** <br><br>


$$R = \begin{bmatrix} r_{11} & r_{12} & r_{13} \\ r_{21} & r_{22} & r_{23} \\ r_{31} & r_{32} & 3_{33} \end{bmatrix} = \begin{bmatrix} \frac{L(x_0 L_9-L_1)}{c_x} & \frac{L(y_0 L_9-L_5)}{c_y} & L*l_9 \\
                    \frac{L(x_0 L_{10}-L_2)}{c_x} & \frac{L(y_0 L_{10}-L_6)}{c_y} & L*l_{10} \\
                    \frac{L(x_0 L_{11}-L_3)}{c_x} & \frac{L(y_0 L_{11}-L_7)}{c_y} & L*l_{11} \\ \end{bmatrix}$$
<br><br>
Lage des Projektionszentrum:<br>
$$ P' =  - \begin{bmatrix} X_0 \\ Y_0 \\ Z_0 \end{bmatrix} = \begin{bmatrix} L_1 & L_2 & L_3 \\ L_5 & L_6 & L_7 \\ L_9 & L_{10} & L_{11} \end{bmatrix}^{-1} \begin{bmatrix} L_4 \\ L_8 \\ 1 \end{bmatrix}$$

Nutze den docstring der Funktion ``direct_linear_transform`` in der Datei [src/direct_linear_transform.py](src/direct_linear_transform.py) um die Übergabeparameter ``img_pts`` und ``obj_pts`` korrekt zu definieren.

In [8]:
img_pts = np.column_stack((x_measured_mm, y_measured_mm))
obj_pts = obj_pts_m[:, 1:4]

In [17]:
from src.direct_linear_transform import *

## **Schritt 3**

#### Berechnung Näherungswerte mittels DLT

Rufe die Funktion ``direct_linear_transform`` mit den zuvor definierten Übergabeparametern auf und vergleiche die Näherungswerte der äusseren Orientierung mit der Übung Einzelbildorientierung aus dem Modul 3030.1 Photogrammetrie.

In [18]:
R, pos_proj_center, interior_orientation = direct_linear_transform(img_pts, obj_pts)

# direct_linear_transform returns the projection centre as a 3x1 column vector
# because the unit tests expect this shape. For printing and iteration we use a flat vector.
pos_proj_center = pos_proj_center.ravel()

x0, y0, cx, cy = interior_orientation
pd = (cx + cy) / 2   # principal distance: use the mean of cx and cy


# Ausgabe in Kommandofenster
print('Näherungswerte aus DLT\n')
print('Parameter der inneren Orientierung')
print('ck = {0:.3f}mm  x0 = {1:.3f}mm  y0 = {2:.3f}mm\n '.format(pd, x0, y0))
print('Parameter der äusseren Orientierung')
print('X0 = {0:.3f}m  Y0 = {1:.3f}m  Z0 = {2:.3f}m  Omega = {3:.3f}°  Phi = {4:.3f}°  Kappa = {5:.3f}°\n '
      .format(*pos_proj_center.tolist(), *rotation_matrix_to_euler(R, full_circle=360)))


Näherungswerte aus DLT

Parameter der inneren Orientierung
ck = 8.642mm  x0 = -0.012mm  y0 = -0.315mm
 
Parameter der äusseren Orientierung
X0 = 63.617cm  Y0 = 226.568cm  Z0 = 111.753cm  Omega = -37.210°  Phi = 23.995°  Kappa = 151.477°
 


Fülle die foldenden Elemente [X0, Y0, Z0, omega, phi, kappa] in die Variable ``projection_center`` für die weiteren Schritte ab.

In [19]:
angles = rotation_matrix_to_euler(R)

projection_center = np.array([
    pos_proj_center[0],
    pos_proj_center[1],
    pos_proj_center[2],
    angles[0],
    angles[1],
    angles[2]
], dtype=float)


## **Schritt 4**
#### Berechnung äussere Orientierung mittels räumlichem Rückwärtsschnitt
(analog zu Übung Einzelbildorientierung Modul 3030.1 Photogrammetrie)

Definiere die Anzahl Iterationen = 10. Instanziere für die vermittelnde Ausgleichung nach der Methode der kleinsten Quadrate die A-Matrix und den f-Vektor basierend auf den Anzahl unbekannten Parameter und der Anzahl Objektpunkten.

In [21]:
num_it = 10

# Instantiate matrices
A = np.zeros((num_pts * 2, 6))
f = np.zeros((num_pts * 2, 1))

# for visualization purposes only
x_corr = np.zeros((num_it, 6))
x_calc = np.zeros((num_pts, num_it))
y_calc = np.zeros((num_pts, num_it))

#### Iterative Berechnung des Rückwärtseinschnitts mittels vermittelnder Ausgleichung

Für die Ausgleichung von nicht linearen Gleichungssysteme ist die Linearisierung der Beobachtungsgleichungen unerlässlich. Dies wurde mit der Berechnung der partiellen Ableitungen nach den Parametern der äusseren Orientierung (Differentialkoeffizienten) bereits implementiert (siehe Übung 3030.1 Einzelbildorientierung). Die linearisierten Beobachtungsgleichungen sind jedoch nur für einen kleinen Bereich um die aktuellen Näherungswärte herum ausreichend genau. Die Ausgleichung wird deshalb iterativ durchgeführt und nähert sich dadurch nach und nach an die korrekte Lösung an.

Im folgenden Code wird die iterative Berechnung des Rückwärtsschnitts durchgeführt. In einer ersten ``for``-Schleife wird über die Anzahl Iterationen iteriert. Innerhalb dieser ``for``-Schleife wird mit einer zweiten ``for``-Schleife über die Bildbeobachtungen iteriert. Dabei werden basierend auf den aktuellen Näherungswerte der äusseren Orientierung die Bildkoordinaten sowie die Differentialkoeffiezienten berechnet und in die A-Matrix bzw. in den f-Vektor abgefüllt.
Anschliessend werden mit der folgenden Formel die Korrekturen für die Näherungswerte $x$ berechnet und die Parameter der äusseren Orientierungen aktualisiert. Zusätzlich werden werden die aktualisierten Werte der äusseren Orientierung zur Plausibilitätkontrolle jeweils ausgegeben.

$$ x = (A^TA)^{-1} * A^Tf$$ 

In der geodätischen Statistik wurde die folgende Formel für die vermittelnde Ausgleichung nach der Methode der kleinsten Quadrate eingeführt:

$$ x = (A^TPA)^{-1} * A^TPf$$ 

Kannst du dir erklären warum die P-Matrix verschwunden ist? 

**Eine kurze Wiederholung / Rekapitlation schadet nie!** Schau dir den Code an und diskutiere ihn mit deinen Komiliton*innen.

In [ ]:
#P-Matrix ist verschwunden, weil hier bei der Messung der Punkten aus dem Bild für jede Messung dieselbe Genauigkeit angenommen wird.
# T Matrix macht dasselbe die wie P matrix

for iteration in range(0, num_it):
    R = rotation_matrix(*projection_center[3:])

    for i in range(0 , u.shape[0]):
        # calculate image coordinates
        x, y = collinearity_equation(X[i], Y[i], Z[i], *projection_center[:3], x0, y0, pd, R)

        # create A matrix and l vector
        A[2 * i : 2 * i + 2 , :] = np.array(part_derivatives_coll_equation(X[i], Y[i], Z[i], *projection_center[:3], pd, projection_center[5], R)).reshape(2,6)
        f[2 * i : 2 * i + 2, : ] = np.array([[x_measured_mm[i] - x],[y_measured_mm[i] - y]])

        # for visualization purposes only
        x_calc[i, iteration] = x
        y_calc[i, iteration] = y


    # Adjustment by least squares method
    x = np.linalg.solve(A.T @ A, A.T @ f)

    # Update prior values by estimated correction term for next iteration and display them
    projection_center += x.ravel()
    display_values(iteration, *projection_center, digits=6)

    # for visualization purposes only
    x_corr[iteration, : ] = x.ravel()

1. Iteration
X0 = 63.709063cm  Y0 = 226.513756cm  Z0 = 111.719893cm  Omega = -37.187035°  Phi = 24.108155°  Kappa = 151.486198°
2. Iteration
X0 = 63.71685cm  Y0 = 226.524732cm  Z0 = 111.707552cm  Omega = -37.207105°  Phi = 24.117735°  Kappa = 151.48815°
3. Iteration
X0 = 63.717538cm  Y0 = 226.525648cm  Z0 = 111.706503cm  Omega = -37.208803°  Phi = 24.118572°  Kappa = 151.488336°
4. Iteration
X0 = 63.717598cm  Y0 = 226.525729cm  Z0 = 111.706411cm  Omega = -37.208951°  Phi = 24.118645°  Kappa = 151.488352°
5. Iteration
X0 = 63.717603cm  Y0 = 226.525736cm  Z0 = 111.706403cm  Omega = -37.208964°  Phi = 24.118652°  Kappa = 151.488353°
6. Iteration
X0 = 63.717604cm  Y0 = 226.525737cm  Z0 = 111.706403cm  Omega = -37.208965°  Phi = 24.118652°  Kappa = 151.488353°
7. Iteration
X0 = 63.717604cm  Y0 = 226.525737cm  Z0 = 111.706403cm  Omega = -37.208965°  Phi = 24.118652°  Kappa = 151.488354°
8. Iteration
X0 = 63.717604cm  Y0 = 226.525737cm  Z0 = 111.706403cm  Omega = -37.208965°  Phi = 24.118652°

## **Schritt 5**

### Visualisierung Resultate für Plausibilitätskontrolle

#### Plot Darstellung, Rückwärtsprojektion der berechneten Koordinaten

Zur Kontrolle werden die gemessenen Sensorkoordinaten sowie die Bildkoordinaten berechnet aus dem Rückwärtseinschnitt gemeinsam im Bild dargestellt. Dazu müssen die Bildkoordinaten jedoch vorgängig in Sensorkoordinaten umgewandelt werden. Führe diese Umrechnung durch. Es resultiert eine Grafik in der die gemessenen Sensorkoordinaten rot und die berechneten Bildkoordinaten je nach Iteration eingefärbt sind.

In [ ]:
plt.figure(figsize=(15,10))
plt.imshow(img)

color_list = ['b', 'g', 'c', 'y', 'y', 'y', 'y', 'y', 'y', 'y']

# Offset point label
offset = 40

# convert image coordinates to sensor coordinates
u_calc = x_calc / pix_size_mm + size_u/2
v_calc = - y_calc  / pix_size_mm + size_v/2

# display calcualted point coordinates from iteration 1 - 10
for i in range (0, num_it):
        plt.scatter(u_calc[:, i], v_calc[:, i], s = 100, facecolors='none', edgecolors=color_list[i])
        add_text(u_calc[:, i], v_calc[:, i], pt_nr, offset, color_list[i], fontsize=13)


# Add ground truth coordinates in red
plt.scatter(u, v, s = 100, facecolors='none', edgecolors='r')
add_text(u,v,pt_nr, offset, color='r', fontsize=13)


#### Konvergenzplot

Um die Konvergenz über sämtliche Iterationen hinweg zu beurteilen, werden die Verbesserungen des unbekannten Vektors $x$ pro Unbekannte und Iteration visualisiert. Im resultierenden Konvergenzplot kann beispielsweise erkannt werden, wenn ein bestimmter Parameter langsamer oder gar nicht konvergiert, was auf grobe Fehler oder systematische Abweichungen hindeuten kann.

In [27]:
plt.figure(figsize=(15,10))

x_axis = np.arange(num_it) + 1

labels = ['X', 'Y', 'Z', 'Omega', 'Phi', 'Kappa']
colors = ['r', 'g', 'b', 'y', 'm', 'c']

for i in range(6):
    plt.plot(x_axis, x_corr[:,i], color = colors[i], label = labels[i])

plt.xlim((1,10))
plt.title("Convergence plot for single image orientation", fontsize = 20)
plt.xlabel("Number of iterations", fontsize = 16)
plt.ylabel("Differences", fontsize = 16)
plt.legend(fontsize=12)

plt.show()